In [1]:
import os
os.chdir("D:/Projects/Skincare_Analyzer")

import pandas as pd
import ast

# 1. Load product_info.csv
df = pd.read_csv("data/raw/product_info.csv")

# 2. Filter to Skincare only (with .copy())
skincare_df = df[df["primary_category"]=='Skincare'].copy()

# 3. Drop rows where ingredients is missing
skincare_df = skincare_df.dropna(subset = ['ingredients'])

print(skincare_df.shape)

(2286, 27)


In [2]:
cols_to_drop = [
    'variation_type', 'variation_value', 'variation_desc',
    'child_count', 'child_max_price', 'child_min_price',
    'value_price_usd', 'sale_price_usd', 'out_of_stock',
    'limited_edition', 'new', 'online_only', 'sephora_exclusive'
]

skincare_df = skincare_df.drop(cols_to_drop, axis=1)  # 1-> col, 0-> row

In [3]:
skincare_df.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'ingredients', 'price_usd', 'highlights',
       'primary_category', 'secondary_category', 'tertiary_category'],
      dtype='object')

In [4]:
def parse_ingredients(text):
    # removing [] -> square brackets
    text = text.strip('[]')
    # splitting at comma
    ingredients = text.split(',')
    # loop through the list, strip spaces AND lowercase each item
    cleaned = [i.strip().lower() for i in ingredients]
    return cleaned

In [5]:
# Applying it to every row
skincare_df['ingredient_list'] = skincare_df['ingredients'].apply(parse_ingredients)

# Check the ingredient list
print(skincare_df['ingredient_list'].iloc[0])

["'collagen (vegan)*", 'water (aqua', 'eau)', 'ethylhexyl palmitate', 'oryza sativa (rice) bran extract', 'caprylic/capric triglyceride', 'glycerin', 'cetearyl methicone', 'dimethicone', 'cetearyl alcohol', 'pyrus malus (apple) fruit extract', 'chlorella protothecoides oil', 'polysorbate 60', 'glyceryl glucoside', 'polyglyceryl-2 stearate', 'parachlorella beijerinckii exopolysaccharides', 'collagen amino acids (vegan)*', 'ceramide np', 'silybum marianum fruit extract', 'helianthus annuus (sunflower) extract', 'rosmarinus officinalis (rosemary) leaf extract', 'tocopherol', 'carnosine', 'sodium stearoyl glutamate', 'caprylyl glycol', 'ammonium acryloyldimethyltaurate/vp copolymer', 'ethylhexylglycerin', 'hexylene glycol', 'citric acid', 'lecithin', 'sodium hydroxide', 'sodium phytate', 't-butyl alcohol', 'leuconostoc/radish root ferment filtrate', 'phenoxyethanol', 'triethyl citrate', 'vanillin', 'amyl salicylate', 'benzyl acetate', 'cedrus atlantica bark oil', 'phenethyl alcohol', 'iono

In [6]:
import re

def parse_ingredients(text):
    # Step 1 — remove leading/trailing quotes and brackets
    text = text.strip().strip("'\"[]")
    
    # Step 2 — smart split: split on commas NOT inside parentheses
    # This regex splits on commas only when not inside ()
    ingredients = re.split(r',\s*(?![^()]*\))', text)
    
    # Step 3 — clean each ingredient
    cleaned = []
    for i in ingredients:
        i = i.strip()           # remove whitespace
        i = i.strip("'\"")      # remove stray quotes
        i = i.rstrip('.')       # remove trailing periods
        i = i.replace('*', '')  # remove * (organic marker)
        i = i.lower()           # lowercase
        i = i.strip()           # final whitespace cleanup
        if i:                   # only add if not empty
            cleaned.append(i)
    
    return cleaned

In [7]:
skincare_df['ingredient_list'] = skincare_df['ingredients'].apply(parse_ingredients)
print(skincare_df['ingredient_list'].iloc[0])

['collagen (vegan)', 'water (aqua, eau)', 'ethylhexyl palmitate', 'oryza sativa (rice) bran extract', 'caprylic/capric triglyceride', 'glycerin', 'cetearyl methicone', 'dimethicone', 'cetearyl alcohol', 'pyrus malus (apple) fruit extract', 'chlorella protothecoides oil', 'polysorbate 60', 'glyceryl glucoside', 'polyglyceryl-2 stearate', 'parachlorella beijerinckii exopolysaccharides', 'collagen amino acids (vegan)', 'ceramide np', 'silybum marianum fruit extract', 'helianthus annuus (sunflower) extract', 'rosmarinus officinalis (rosemary) leaf extract', 'tocopherol', 'carnosine', 'sodium stearoyl glutamate', 'caprylyl glycol', 'ammonium acryloyldimethyltaurate/vp copolymer', 'ethylhexylglycerin', 'hexylene glycol', 'citric acid', 'lecithin', 'sodium hydroxide', 'sodium phytate', 't-butyl alcohol', 'leuconostoc/radish root ferment filtrate', 'phenoxyethanol', 'triethyl citrate', 'vanillin', 'amyl salicylate', 'benzyl acetate', 'cedrus atlantica bark oil', 'phenethyl alcohol', 'ionone', 

In [8]:
def parse_highlights(text):
    try:
        result = ast.literal_eval(text)
        return result                     
    except:
        return []                        # fallback if something breaks

In [9]:
skincare_df['highlights_list'] = skincare_df['highlights'].apply(parse_highlights)
print(skincare_df['highlights_list'].iloc[0])

['Vegan', 'Good for: Loss of firmness', 'Collagen', 'Hypoallergenic', 'Without Parabens', 'Best for Dry, Combo, Normal Skin']


In [10]:
# Save cleaned df
skincare_df.to_csv("data/processed/skincare_df.csv", index=False) # to avoid first column as index
print("DF saved")

DF saved


In [11]:
skincare_df.shape

(2286, 16)

In [12]:
skincare_df.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'ingredients', 'price_usd', 'highlights',
       'primary_category', 'secondary_category', 'tertiary_category',
       'ingredient_list', 'highlights_list'],
      dtype='object')